# ASL-COLAB-02: Unsloth inference notebook (single-gloss + deterministic OOV retry)

This notebook is the Colab-first execution slice for issue #89.
It enforces:
- single Top-50 gloss output contract
- one deterministic OOV retry with stronger constraints
- per-sample first-pass vs retry logging


In [1]:
import os
os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'


## Install deps (Colab)
Uncomment when running in a fresh Colab runtime.


In [2]:
!pip -q install unsloth transformers peft accelerate pandas huggingface_hub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 40.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 170.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 130.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 137.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 56.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 135.6 MB/s eta 0:00:00
   ━━━━

## Colab setup prerequisites
For a clean demo run, do this first:
1) Clone this repo into Colab and run notebook from repo root so `from src...` imports work.
2) Ensure data files exist at the configured paths (`MANIFEST`, `TEST_JSONL`).
3) If checkpoint is private on HF, run `from huggingface_hub import login; login()` once.


In [3]:
from pathlib import Path
import json
import pandas as pd

from src.evaluation.colab_unsloth_inference import evaluate_anchor_samples
from src.evaluation.colab_anchor_contract import build_colab_anchor_contract
from src.evaluation.unsloth_asl import RealUnslothASLGlossPredictor, normalize_gloss
from src.evaluation.colab_gatekeeper import evaluate_colab_gatekeeper


ModuleNotFoundError: No module named 'src'

## Configure paths
Update these paths for your Colab drive/repo layout.


In [ ]:
CHECKPOINT = 'checkpoints/asl-gemma4-e2b-q64-top50-merged-16bit'
MANIFEST = 'data/processed/exports/asl_unsloth_pose_train_q64_full_top50_manifest.json'
TEST_JSONL = 'data/processed/exports/asl_unsloth_pose_train_q64_full_top50_test.jsonl'
OUT_DIR = Path('evaluation/results/colab_issue89_single_gloss_retry')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# If you only have an HF repo id, download snapshot first:
# from huggingface_hub import snapshot_download
# CHECKPOINT = snapshot_download('AlexD281/asl-gemma4-e2b-q64-top50-merged-16bit')


In [ ]:
contract = build_colab_anchor_contract(
    manifest_path=MANIFEST,
    records_path=TEST_JSONL,
    top_k=10,
)
allowlist = contract['allowlist']
alias_entries = contract['alias_map']
anchor_glosses = [row['gloss'] for row in contract['anchors']]

print('allowlist:', len(allowlist))
print('anchors:', anchor_glosses)


In [ ]:
# Build anchor sample set: first 3 examples per anchor gloss (deterministic order by file order)
raw_records = []
with open(TEST_JSONL, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            raw_records.append(json.loads(line))

samples = []
counts = {g: 0 for g in anchor_glosses}
for row in raw_records:
    expected = normalize_gloss(str(row.get('output','')))
    if expected in counts and counts[expected] < 3:
        counts[expected] += 1
        samples.append({
            'sample_id': str(row.get('sample_id') or row.get('id') or f"row-{len(samples)}"),
            'expected_gloss': expected,
            'input': str(row.get('input','')),
        })

print('selected samples:', len(samples))
print('per-anchor counts:', counts)


In [ ]:
predictor = RealUnslothASLGlossPredictor(CHECKPOINT, max_new_tokens=4)

def predict_fn(prompt: str, decode: dict):
    # deterministic decode contract
    predictor.max_new_tokens = int(decode.get('max_new_tokens', 4))
    return predictor.predict_raw({'instruction': '', 'input': prompt})


In [ ]:
rows = evaluate_anchor_samples(
    samples=samples,
    allowlist=allowlist,
    alias_entries=alias_entries,
    predict_fn=predict_fn,
)

df = pd.DataFrame(rows)
metrics = {
    'sample_count': int(len(df)),
    'accuracy': float(df['correct'].mean()) if len(df) else 0.0,
    'retry_rate': float(df['retry_used'].mean()) if len(df) else 0.0,
    'oov_after_retry_rate': float((~df['final_valid']).mean()) if len(df) else 0.0,
}

pred_csv = OUT_DIR / 'colab_issue89_predictions.csv'
metrics_json = OUT_DIR / 'colab_issue89_metrics.json'

df.to_csv(pred_csv, index=False)
metrics_json.write_text(json.dumps(metrics, indent=2) + '
', encoding='utf-8')

print(metrics)
print('wrote', pred_csv)
print('wrote', metrics_json)


## Acceptance checks
- Output contract enforced by `final_valid` in the CSV.
- One deterministic retry logged via `retry_used`, `first_pass_raw`, `retry_raw`.
- Metrics and predictions exported for issue evidence.


## Demo gatekeeper (issue #90)
Run the go/no-go decision directly in Colab after inference.


In [ ]:
gate_report = evaluate_colab_gatekeeper(
    contract=contract,
    prediction_rows=rows,
    per_anchor_threshold=0.70,
    collapse_threshold=0.40,
    anchor_min_samples=3,
    anchor_max_samples=5,
)

gate_json = OUT_DIR / 'colab_issue90_gatekeeper.json'
gate_json.write_text(json.dumps(gate_report, indent=2) + '\n', encoding='utf-8')
print('wrote', gate_json)


In [ ]:
final_pass = bool(gate_report['final_after_retry']['pass'])
first_pass = bool(gate_report['first_pass']['pass'])
changed = bool(gate_report['retry_effect']['decision_changed'])

print('=== COLAB DEMO GATE ===')
print('FINAL_GATE:', 'PASS' if final_pass else 'FAIL')
print('FIRST_PASS:', 'PASS' if first_pass else 'FAIL')
print('RETRY_CHANGED_DECISION:', changed)

if gate_report['final_after_retry']['reasons']:
    print('FAIL_REASONS:')
    for reason in gate_report['final_after_retry']['reasons']:
        print('-', reason)

per_anchor_df = pd.DataFrame(gate_report['final_after_retry']['per_anchor'])
per_anchor_df
